# SmartPlate Model Fine-Tuning Pipeline

This notebook fine-tunes a Large Language Model for the **SmartPlate** AI assistant using PEFT (LoRA) and 4-bit quantization (bitsandbytes). It prepares the dataset, configures memory-efficient training with `SFTTrainer`, and saves the adapter checkpoints directly to Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


1. Dependency Installation

In [ ]:
!pip install -U -q transformers peft trl bitsandbytes accelerate datasets "pyarrow>=19.0.0,<24"

In [ ]:
import os
import torch

# Compatibilité anciennes versions de PyTorch
if not hasattr(torch, "float8_e8m0fnu"):
    setattr(torch, "float8_e8m0fnu", torch.float32)

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from collections import Counter

In [ ]:
# ==============================================================================
# GPU : la T4 (Colab gratuit) NE supporte PAS bf16 -> on utilise fp16.
# ==============================================================================
assert torch.cuda.is_available(), "Pas de GPU : Exécution > Modifier le type d'exécution > T4 GPU"
use_bf16 = torch.cuda.get_device_capability()[0] >= 8
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
print(f"GPU : {torch.cuda.get_device_name(0)} | précision : {'bf16' if use_bf16 else 'fp16'}")

# ==============================================================================
# Chemins (Google Drive)
# ==============================================================================
BASE_DIR = "/content/drive/MyDrive/Fine_Tunning"
dataset_path = f"/content/drive/MyDrive/Fine_Tunning/Data/smart_plate_nutrition_dataset_clean.jsonl"
DOSSIER_SAUVEGARDE_MODELE = f"/content/drive/MyDrive/Fine_Tunning/smart_plate_nutrition_model_finetuned"

if not os.path.exists(dataset_path):
    raise FileNotFoundError(
        f"Dataset introuvable : {dataset_path}\n"
        "Vérifie que Drive est monté et que le fichier .jsonl est bien à cet endroit."
    )
os.makedirs(DOSSIER_SAUVEGARDE_MODELE, exist_ok=True)

model_id = "Qwen/Qwen3-4B-Base"

# ==============================================================================
# Tokenizer
# ==============================================================================
print("Chargement du tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = "right"

tokenizer.eos_token = "<|endoftext|>"

# Correction : Utilisation d'un token de padding standard pour un modèle textuel
tokenizer.pad_token = tokenizer.eos_token

# Template simple
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}{{ 'Système :\n' + message['content'] + '\n\n' }}"
    "{% elif message['role'] == 'user' %}{{ 'Utilisateur :\n' + message['content'] + '\n\n' }}"
    "{% elif message['role'] == 'assistant' %}{{ 'Assistant :\n' + message['content'] + '<|endoftext|>' }}"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ 'Assistant :\n' }}{% endif %}"
)

# ==============================================================================
# Dataset
# ==============================================================================
print(f"Chargement du dataset depuis : {dataset_path}")
dataset = load_dataset("json", data_files=dataset_path, split="train")

def to_prompt_completion(example):
    msgs = example["messages"]
    return {"prompt": msgs[:-1], "completion": msgs[-1:]}

dataset = dataset.map(to_prompt_completion, remove_columns=dataset.column_names)

# Vérification visuelle
example_text = tokenizer.apply_chat_template(
    dataset[0]["prompt"] + dataset[0]["completion"], tokenize=False
)
print("\n--- Exemple formaté ---")
print(example_text)
print("--- Fin de l'exemple ---\n")
assert "<think>" not in example_text, "Le template ajoute un bloc <think>."
assert "<|im_start|>" not in example_text, "Des tokens ChatML sont présents."
assert example_text.endswith("<|endoftext|>"), "L'exemple ne finit pas par <|endoftext|>."

# ==============================================================================
# Modèle 4-bit (QLoRA)
# ==============================================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

print("Chargement du modèle en 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,  # CORRECTION CLÉ : force le float16 pour éviter le conflit BFloat16/GradScaler
)
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# ==============================================================================
# LoRA
# ==============================================================================
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# ==============================================================================
# Config d'entraînement
# ==============================================================================
training_args = SFTConfig(
    output_dir=DOSSIER_SAUVEGARDE_MODELE,
    max_length=1024,
    completion_only_loss=True,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=1,
    optim="paged_adamw_8bit",
    bf16=use_bf16,
    fp16=not use_bf16,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
)

b = trainer.data_collator([trainer.train_dataset[0]])
print(b["labels"][0][-3:], tokenizer.eos_token_id)
def dtypes_entrainables():
    return Counter(str(p.dtype) for p in trainer.model.parameters() if p.requires_grad)

print("dtypes entraînables avant :", dtypes_entrainables())
for p in trainer.model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.to(torch.float32)
print("dtypes entraînables après :", dtypes_entrainables())


# Reprise automatique si la session Colab s'est coupée
has_checkpoint = any(
    d.startswith("checkpoint-") for d in os.listdir(DOSSIER_SAUVEGARDE_MODELE)
)
print("\nDébut de l'entraînement..." + (" (reprise depuis le dernier checkpoint)" if has_checkpoint else ""))
trainer.train(resume_from_checkpoint=has_checkpoint)

print(f"\nEntraînement terminé ! Sauvegarde dans : {DOSSIER_SAUVEGARDE_MODELE}")
trainer.model.save_pretrained(DOSSIER_SAUVEGARDE_MODELE)
tokenizer.save_pretrained(DOSSIER_SAUVEGARDE_MODELE)

GPU : Tesla T4 | précision : fp16
Chargement du tokenizer...
Chargement du dataset depuis : /content/drive/MyDrive/Fine_Tunning/Data/smart_plate_nutrition_dataset_clean.jsonl

--- Exemple formaté ---
Système :
Tu es l'assistant de 'Smart Plate Nutrition'. Ton rôle est de fournir des informations nutritionnelles ultra-précises, scientifiques et personnalisées. Tu dois impérativement t'exprimer avec une grande politesse, utiliser rigoureusement le vouvoiement, commencer par une salutation chaleureuse et terminer en encourageant l'utilisateur de manière bienveillante.

Utilisateur :
Voici ce que je vais manger : 150g de riz blanc cuit, 250g de salade verte et 150g de patate douce rôtie. Pouvez-vous me donner les calories et macros ?

Assistant :
Bonjour ! C'est un réel plaisir de vous accompagner aujourd'hui.

Voici l'analyse détaillée de votre assiette :

- **Riz blanc cuit (150g) :** env. 195 kcal | P: 4.1g | G: 42.0g | L: 0.4g
- **Salade verte (250g) :** env. 38 kcal | P: 3.5g | G: 7.2

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

tensor([ 84715,    753, 151643]) 151643
dtypes entraînables avant : Counter({'torch.bfloat16': 504})
dtypes entraînables après : Counter({'torch.float32': 504})

Début de l'entraînement...


Step,Training Loss
10,0.793035
20,0.387619
30,0.282876
40,0.228717
50,0.202150
60,0.210913
70,0.180724
80,0.180961
90,0.186062
100,0.167938



Entraînement terminé ! Sauvegarde dans : /content/drive/MyDrive/Fine_Tunning/smart_plate_nutrition_model_finetuned


('/content/drive/MyDrive/Fine_Tunning/smart_plate_nutrition_model_finetuned/tokenizer_config.json',
 '/content/drive/MyDrive/Fine_Tunning/smart_plate_nutrition_model_finetuned/chat_template.jinja',
 '/content/drive/MyDrive/Fine_Tunning/smart_plate_nutrition_model_finetuned/tokenizer.json')